In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL", "gemini-2.0-flash"),
    google_api_key=os.getenv("GEMINI_API_FREE_KEY") or os.getenv("GEMINI_API_KEY")
)

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content=[{'type': 'text', 'text': '네, 잘 지내고 있었어요! 물어봐 주셔서 감사합니다. 😊\n\n사용자님도 그동안 잘 지내셨나요? 오늘 하루는 어떠셨는지, 혹은 제가 도와드릴 일이 있는지 궁금하네요! 무엇이든 편하게 이야기해 주세요.', 'extras': {'signature': 'EvIQCu8QAb4+9vv83MGrT+tO1PcY1Zoof/fALOEy7S+HpPPL5ZCKAzpmIXys7pHLx7/Cww4d/iXtb5Tspuiu8h/uOy+ABZp8v0k02WG4BpJiRiURyV0z/gfWqUBZsIgLlbiZgdg5U+o6dSZmo4MxdDw9at+H9hYyOjS550KPZ3aKFUSfYqx1tZV4/dN1r5WP9pDHnSdBPQ8ZPffv6JiheB8dYKEkw9ShJzUJfktouGD+KM7eX8xwFyr2y6hIFJTT78pp01b2OfIy18t0TzXjQnKBpEGVa7a3o9UrLH/cUoIfn4jC0imws5tHvKf9xQ1oONOZpSzoe5oG5zbVwPWU+C/pS5aY+rsfUN8xp8LJLN6GaRi86vdpNnLoSj0zHRgckXFQvmkGn+utI0tAPG6+nuNrAntApLFuGTihC0FJZoPVyJss2hsCmRK1gvZBtNYjEp7wbViG6MHSoObzDwqm4gY3XZkMWeTgN93E4V3yv1lBu0U4yO531tkWRG+mAuKwRgvcvfvaYQZanoOcU9VFLFLXIcdmIygTcwOoW0QP/cYhwq7rtxnp4KkoFztof91qD+2hM0Mw1K2zdz0aCTg7tpAN/5h3hIzidD6vxOJ/uLeMZ9RguYaGUxR+TRHzitJth1pLJABToEyTKTFUyfXUXjorG3STdjjkptFu5tb1dY2D9AHOkanFfwuLnzZB/bkMRsuWWSDwNS8BqRRL4iggmHL0/4fYK3lGkd3zFb5QCR8g0e+ZnShQuC8YDSJPwXF8PEdMXsZLoxNwwAv1J73D1i0z3wQx7+OihgNrKV8K3qgQGAhf

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [3]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}'}, '__gemini_function_call_thought_signatures__': {'e7245a1e-6452-4d81-a20f-e44b61077028': 'EuQBCuEBAb4+9vu0/TwSlW53USF78ohqrDIiakrJVD04pqJuGozgnNCtQk6cySKdxkrBnAxKOEAcJsQzLfIhzej0AxM+SxXtK/ZQLgMALiHcJRtSDkCfJgEgpqJ9q55e9a6v1yQTQwtcwuaNkfY3I4WiylPMZ78FAApR02DCSKMznMM0CGblOJNvzQ6oKs4+E3xPerId/TfvvCKKFOTCtGx95TEzg+O6t5aKjgN8Y9xAxXJXwX3GENxFAV7PUBgE8MUzfnGWfmZtsXdjv2R1cEew8ekHW5ttXlQIY0HEz5iTsk2nLvg7'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d1848-2ccf-7d42-aac4-f411bf0e5024-0', tool_calls=[{'name': 'get_current_time', 'args': {'t

In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-03-23 10:21:38 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}'}, '__gemini_function_call_thought_signatures__': {'e7245a1e-6452-4d81-a20f-e44b61077028': 'EuQBCuEBAb4+9vu0/TwSlW53USF78ohqrDIiakrJVD04pqJuGozgnNCtQk6cySKdxkrBnAxKOEAcJsQzLfIhzej0AxM+SxXtK/ZQLgMALiHcJRtSDkCfJgEgpqJ9q55e9a6v1yQTQwtcwuaNkfY3I4WiylPMZ78FAApR02DCSKMznMM0CGblOJNvzQ6oKs4+E3xPerId/TfvvCKKFOTCtGx95TEzg+O6t5aKjgN8Y9xAxXJXwX3GENxFAV7PUBgE8MUzfnGWfmZtsXdjv2R1cEew8ekHW5ttXlQIY0HEz5iTsk2nLvg7'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d1848-2ccf-7d42-aac4-f411bf0e5024-0', tool_calls=[{'name': 'get_current_time', 'args': {

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content=[{'type': 'text', 'text': '부산의 현재 시각은 2026년 3월 23일 오전 10시 21분입니다.'}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d1848-aa70-78c1-9f3b-30b015ef5410-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 263, 'output_tokens': 30, 'total_tokens': 293, 'input_token_details': {'cache_read': 0}})

In [7]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")


In [8]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [9]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content=[] additional_kwargs={'function_call': {'name': 'get_yf_stock_history', 'arguments': '{"stock_history_input": {"ticker": "TSLA", "period": "1mo"}}'}, '__gemini_function_call_thought_signatures__': {'2a634db8-cca5-4f09-b94b-f8bd41a99ca9': 'EpoECpcEAb4+9vuhN6ecZeE70KkYA2fRtSG9tCFkyDXj+lyU9vJt3VXJtFKiMWew5V/vzv+mHo0wg7IHzOlL4+IOpJUcXjI1ImcVzuM3TCNpJ7BtUZfpvGb22K1cNkEt7aGmz2wMGj87E2NV6HeceFVwz3Hp8o740tjjyQXVYBWsczz/omHQRoNMQjFEveWbic4a+k+w5zSULqZE6eWRRWPGH3A46XQiNexAwy6Q82WCWQ7jA1f5UdgGvB+HlZ4Ux4lyvM8BrwT2uoav7+45XKo0jsyF+gAGccF7sLT0teIsVFzO5W+394ATmDlPvNYKNEPlIDxF30Zyto+LN2tQEk8KSQsqnwWaLTs3UCku7ODF/nm25iREW4E6BWVPodmeBjHwpOj1xgr8BAAj/sm0A9LOj3y0iCxwT54vV+y5NbF557chSDfTNWncFceL2UBr+U6awQPLI2TRkq+Snca+MtytRQncUG2g1vKpu5whCEUjNCuLYuTeayh0qvV0AP/ekx6jTyftmC8KP/KiXsWvrjetH4kV6LzskCar+NYYuOkQpfPIkpNIfsVFj6mHdNunqOc2zQIZvuKt6AACkjqtZmpr4sJ2VDFiw5sNw1lztNeF07EHvR6eRpKPqUaqJAPp2jJ1hGcsUXr00hVJdpi4joYPq2FIp3IbHTecqpjDTsJiOxjbeBRW5fkNsuG1oLX+uachUXmU3wJ/RnPWYw=='}} response_metadata={'finis

In [10]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-02-23 00:00:00-05:00 | 407.29 | 407.7  | 394.04 |  399.83 | 6.968e+07   |           0 |              0 |\n| 2026-02-24 00:00:00-05:00 | 399.5  | 410.82 | 397.64 |  409.38 | 5.85795e+07 |           0 |              0 |\n| 2026-02-25 00:00:00-05:00 | 412.15 | 420.34 | 412.15 |  417.4  | 5.48097e+07 |           0 |              0 |\n| 2026-02-26 00:00:00-05:00 | 414.42 | 416.81 | 403.66 |  408.58 | 5.36025e+07 |           0 |              0 |\n| 2026-02-27 00:00:00-05:00 | 402.94 | 407.12 | 398.11 |  402.51 | 5.68901e+07 |           0 |              0 |\n| 2026-03-02 00:00:00-05:00 | 390.6  | 404.54 | 388.25 |  403.32 | 5.50883e+07 |           0 |              0 |\n| 2026-03-03 00:00:00-05:0

In [11]:
llm_with_tools.invoke(messages)

AIMessage(content=[{'type': 'text', 'text': '테슬라(TSLA)의 주가는 한 달 전과 비교했을 때 **하락**했습니다.\n\n*   **한 달 전(2026년 2월 23일) 종가:** 399.83달러\n*   **최근(2026년 3월 20일) 종가:** 367.96달러\n\n약 한 달 사이 주가는 약 **31.87달러(-7.97%) 하락**했습니다.', 'extras': {'signature': 'Es8DCswDAb4+9vuXOhJfJ6+KfHxipIqZ2iSHDDVvjfkoQ3y9WuvKHtVNztgimRnTL2SGo1tOyOsODufrtWVGCdfnRwe+ae9lVBSYgnTCFHB1N0bCHOauc9UQxRB1SQFVaeVfqCcI7pucAJrWCgK7JlETJjii71PBydfDBTzTG2CYGZIZVhKeWbIr2bI7XyZ1ivlaVuf2XpD3azTJtNMiFiYuGsaY+shbXy4YvaCfKHL+FeGEHtYjzzBc/YwwK09NJc9XTo01fN5PorH6B7s4GnDoZhZX7FU+ekmVo8AcxB+jqiu/MoHKjHjo3BVXbL6sfoxp/do2k37VyWYuCsDiBL60NCvHqJmdGCWXoUNitJP+ux6L+8yoPp34PRqANdDvB9jh2OB5j9BNM2UPqAdCW2xOyGnFuVjhiOe2/RDnJ8jKNZnQXw2uswJS5Ycdt6rVFvzjk1IlJOnOVTSDFqr/yS/SmYFjAOeuRGBJzznqUZwShdYhHPP/Ncmxi8fidmAzpqoF/hmT20B3U+G976MkQuDqV6xSe1e87ghPIO+uTlKGUE8JkXxi4mjxQ7iN5vzW+fnxr2AI/7zlH5fp/JrnMhYWJn4HrkTZWTHBj0keWhbemw=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 